# Domain 8 — Tools and MCPs (10.6%)

| Skill | Weight |
|---|---|
| Tool Implementation | 4.4% |
| Agentic Customization | 4.1% |
| MCP Server Development | 2.1% |


In [ ]:
"""Shared setup. Export ANTHROPIC_API_KEY before launching Jupyter."""

import json
import os

import anthropic

client = anthropic.Anthropic()

OPUS = "claude-opus-5"
SONNET = "claude-sonnet-5"
HAIKU = "claude-haiku-4-5-20251001"

MODEL = SONNET


def extract_text(response: anthropic.types.Message) -> str:
    """Concatenate text blocks, ignoring thinking and tool_use blocks."""
    return "".join(
        block.text for block in response.content if block.type == "text"
    )


print("API key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))


## 8.1 Tool Implementation — the description is the prompt

A tool's `description` is how the model decides whether and when to call
it. Vague descriptions cause wrong-tool selection, and the exam tests this
directly. A good description states:

1. What the tool does.
2. The exact input format, with an example.
3. What it returns.
4. When *not* to call it, and how it differs from similar tools.


In [ ]:
# Anti-pattern: the model cannot tell these apart.
WEAK_TOOLS = [
    {
        "name": "search_claims",
        "description": "Search claims.",
        "input_schema": {
            "type": "object",
            "properties": {"q": {"type": "string"}},
            "required": ["q"],
        },
    },
    {
        "name": "get_claim",
        "description": "Get a claim.",
        "input_schema": {
            "type": "object",
            "properties": {"id": {"type": "string"}},
            "required": ["id"],
        },
    },
]

# Preferred: each description says when to use it and when not to.
STRONG_TOOLS = [
    {
        "name": "search_claims",
        "description": (
            "Search claims by free-text criteria such as claimant surname, "
            "peril, or date range, and return up to 20 matching claim IDs "
            "with one-line summaries. Use this when you do NOT already "
            "have a claim ID. Do not use it to fetch details of a known "
            "claim -- call get_claim for that."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": (
                        "Free-text criteria, e.g. 'hail claims filed in "
                        "March 2026' or 'claimant Okafor'."
                    ),
                },
                "limit": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 20,
                    "description": "Max results to return. Defaults to 5.",
                },
            },
            "required": ["query"],
        },
    },
    {
        "name": "get_claim",
        "description": (
            "Retrieve the full record for ONE claim by its exact ID: "
            "status, payout amount, peril, and filing date. Claim IDs have "
            "the format ABC-123. Call once per claim ID. Never invent an "
            "ID -- if you do not have one, call search_claims first."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "claim_id": {
                    "type": "string",
                    "pattern": "^[A-Z]{3}-[0-9]{3}$",
                    "description": "Exact claim ID, e.g. ABC-123.",
                }
            },
            "required": ["claim_id"],
        },
    },
]

print("Compare the two descriptions above before running the next cell.")


In [ ]:
CLAIMS = {
    "ABC-123": {"status": "APPROVED", "amount": 8400, "peril": "hail"},
    "XYZ-789": {"status": "DENIED", "amount": 0, "peril": "flood"},
    "DEF-456": {"status": "PENDING", "amount": 15200, "peril": "fire"},
}


def search_claims(query: str, limit: int = 5) -> str:
    """Naive substring search across the claim records."""
    hits = [
        {"claim_id": claim_id, **record}
        for claim_id, record in CLAIMS.items()
        if query.lower() in json.dumps(record).lower()
    ]
    return json.dumps(hits[:limit])


def get_claim(claim_id: str) -> str:
    """Fetch one claim, or an error string if it does not exist."""
    record = CLAIMS.get(claim_id)
    if record is None:
        return f"ERROR: no claim with id {claim_id}"
    return json.dumps(record)


REGISTRY = {"search_claims": search_claims, "get_claim": get_claim}


def which_tool(tools: list[dict], request: str) -> list[str]:
    """Report which tools the model picks for a request.

    Same request, two tool sets -- the only variable is description
    quality.
    """
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        tools=tools,
        messages=[{"role": "user", "content": request}],
    )
    return [
        f"{block.name}({block.input})"
        for block in response.content
        if block.type == "tool_use"
    ]


request = "Find all the hail claims we have on file."
print("weak tools  ->", which_tool(WEAK_TOOLS, request))
print("strong tools->", which_tool(STRONG_TOOLS, request))


### Client-side vs. server-side tools

- **Client-side** — you define the schema, you execute the function, you
  return a `tool_result`. Every tool in this notebook is client-side.
- **Server-side** — Anthropic executes it (web search, code execution).
  You do not run a loop for it; results come back inside the response, and
  some carry usage-based pricing beyond tokens.

The exam distinction: a client-side tool needs your execution loop; a
server-side tool does not.


In [ ]:
def run_tool_loop(request: str, max_turns: int = 6) -> str:
    """Bounded client-side tool loop over STRONG_TOOLS."""
    messages: list[dict] = [{"role": "user", "content": request}]

    for turn in range(1, max_turns + 1):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=STRONG_TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return extract_text(response)

        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            output = REGISTRY[block.name](**block.input)
            print(f"turn {turn}: {block.name}({block.input})")
            results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                    "is_error": output.startswith("ERROR:"),
                }
            )
        messages.append({"role": "user", "content": results})

    return f"STOPPED after {max_turns} turns."


print(run_tool_loop("Find the hail claims, then give me the full record."))


## 8.2 MCP — host, client, server

The three roles, precisely — and conflating them is a known exam trap:

- **Host** — the application the user interacts with (Claude Code, Claude
  Desktop, your own app). It owns the conversation and decides when tools
  run.
- **Client** — lives inside the host. One client maintains one connection
  to one server, handling the protocol.
- **Server** — exposes capabilities. It is *called*, never the caller.

The thing people get backwards: your application code that calls the Claude
API is the **host**, not a server. The server is a separate process you
connect to.

An MCP server exposes three kinds of capability:

- **Tools** — model-invoked actions.
- **Resources** — readable context, addressed by URI.
- **Prompts** — reusable templates the user invokes.

Run the next cell to write a working server, then run it in a terminal.


In [ ]:
SERVER_SOURCE = '''"""Minimal MCP server exposing claims capabilities.

Run with:  python claims_mcp_server.py
Install with:  pip install mcp
"""

import json

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("claims-server")

CLAIMS = {
    "ABC-123": {"status": "APPROVED", "amount": 8400, "peril": "hail"},
    "XYZ-789": {"status": "DENIED", "amount": 0, "peril": "flood"},
}


@mcp.tool()
def get_claim(claim_id: str) -> str:
    """Retrieve a claim record by its exact ID (format ABC-123).

    The docstring becomes the tool description the model reads, so it
    carries the same weight as a hand-written description field.
    """
    record = CLAIMS.get(claim_id)
    if record is None:
        return f"ERROR: no claim with id {claim_id}"
    return json.dumps(record)


@mcp.resource("claims://policy-manual")
def policy_manual() -> str:
    """Readable context, addressed by URI. Not model-invoked."""
    return "SECTION 1. COVERED PERILS: fire, hail, windstorm, theft."


@mcp.prompt()
def triage_prompt(claim_text: str) -> str:
    """A reusable template the user invokes, not the model."""
    return f"Triage this claim as APPROVE, DENY or REVIEW:\\n{claim_text}"


if __name__ == "__main__":
    # stdio: the host launches this as a subprocess and speaks over
    # stdin/stdout. Best for local, single-user servers.
    # Streamable HTTP is the alternative for remote, multi-client servers.
    mcp.run(transport="stdio")
'''

with open("claims_mcp_server.py", "w") as handle:
    handle.write(SERVER_SOURCE)

print("wrote claims_mcp_server.py")
print("run it:  pip install mcp && python claims_mcp_server.py")


### Transports

- **stdio** — the host launches the server as a subprocess and talks over
  stdin/stdout. Local, single-user, no network surface. The default for
  developer tooling.
- **Streamable HTTP** — the server runs as a network service, serving many
  clients. Needed for remote or shared deployments, and it brings auth and
  network security into scope.

Pick stdio for a local dev tool; pick HTTP when the server must be shared.


## 8.3 Agentic Customization — choosing the mechanism

Four ways to extend Claude. The exam gives a scenario and asks which fits:

| Mechanism | Use when | Runs where |
|---|---|---|
| **Built-in tool** | Anthropic already provides it (web search, code execution) | Anthropic's infrastructure |
| **Custom tool** | One app needs one capability; no reuse elsewhere | Your app's loop |
| **Skill** | Reusable *instructions and procedure*, not an API call | Loaded into context |
| **MCP server** | A capability several apps need, maintained independently | Its own process |

The two decision hinges:

- **Reuse across applications** → MCP server. A custom tool duplicated in
  five codebases is five places to fix a bug.
- **Knowledge versus action** → a Skill teaches Claude *how your team does
  something*; a tool gives it the ability to *do* something external.

Worked answers:

1. *Web search for a research agent* → **built-in tool**. Already exists;
   building your own is wasted work.
2. *One-off CSV reformatting for a single project* → **custom tool** (or a
   Skill if it is procedure rather than code). No reuse, so no MCP.
3. *Inventory lookups needed by five apps, owned by a platform team* →
   **MCP server**. This is the textbook case: shared, independently
   maintained, one place to fix.
4. *"How we write release notes" style guide across many repos* →
   **Skill**. It is procedural knowledge, not an external call.
